# StarCoder2-3B QLoRA Fine-Tuning for Code Completion

A production-oriented notebook for fine-tuning **bigcode/starcoder2-3b** using 4-bit NF4 QLoRA, completion-aware labels, gradient checkpointing, paged 8-bit AdamW, proper held-out evaluation, and completion-only inference.

**Dataset alignment:** Configured for the CodeSearchNet Python dataset located in `dataset/` (`python_train_*.jsonl`, `python_valid_*.jsonl`, `python_test_*.jsonl`).


In [ ]:
# 0. Install dependencies
# !pip install -U "transformers>=4.48" "peft>=0.14" "bitsandbytes>=0.45" "accelerate>=1.2" "datasets>=3.2" "safetensors>=0.4"

In [1]:
# 1. Imports, versions, GPU

import os, glob, math, random, json
import numpy as np
import torch

import transformers, datasets, peft, bitsandbytes as bnb, accelerate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes:", bnb.__version__)
print("Accelerate:", accelerate.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this QLoRA notebook.")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PyTorch: 2.14.0+cu130
Transformers: 5.17.0
Datasets: 5.0.1
PEFT: 0.21.2
bitsandbytes: 0.50.2
Accelerate: 1.15.0
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 7.62 GB


In [2]:
# 2. Configuration

MODEL_NAME = "bigcode/starcoder2-3b"

# Smart BASE_DIR detection works whether notebook is executed from root or notebooks/ folder
cwd = os.getcwd()
if os.path.exists(os.path.join(cwd, "dataset")):
    BASE_DIR = cwd
elif os.path.exists(os.path.join(cwd, "..", "dataset")):
    BASE_DIR = os.path.abspath(os.path.join(cwd, ".."))
else:
    BASE_DIR = cwd

TRAIN_GLOB = os.path.join(BASE_DIR, "dataset", "train", "*.jsonl")
VALID_GLOB = os.path.join(BASE_DIR, "dataset", "valid", "*.jsonl")
TEST_GLOB  = os.path.join(BASE_DIR, "dataset", "test", "*.jsonl")

# ---- DATASET SCHEMA (CodeSearchNet format) ----
# CodeSearchNet dataset in dataset/ has keys:
# [repo, path, func_name, original_string, language, code, code_tokens, docstring, docstring_tokens, sha, url, partition]
#
# Option A: Prompt -> Completion mode (e.g. docstring -> function code)
# PROMPT_COLUMN = "docstring"
# COMPLETION_COLUMN = "original_string" # or "code"
PROMPT_COLUMN = None
COMPLETION_COLUMN = None

# Option B: Full-text Causal LM mode (complete python function string)
# "original_string" contains exact function code with docstrings and comments.
FULL_TEXT_COLUMN = "original_string"

# ---- DATASET SIZE ----
USE_SUBSET = True
TRAIN_SAMPLES = 20_000
EVAL_SAMPLES = 2_000

# ---- CONTEXT ----
MAX_LENGTH = 1024

# ---- QLoRA ----
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# ---- TRAINING ----
NUM_EPOCHS = 2
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03

LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 2

OUTPUT_DIR = os.path.join(BASE_DIR, "checkpoints", "starcoder2-3b-qlora-code-completion")
ADAPTER_DIR = os.path.join(BASE_DIR, "saved_models", "starcoder2-3b-qlora-code-completion")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)


BASE_DIR: /home/mastermind/Desktop/extras/CodeCompletionModel


In [3]:
# 3. Load and inspect JSONL data BEFORE training

train_files = sorted(glob.glob(TRAIN_GLOB))
valid_files = sorted(glob.glob(VALID_GLOB))
test_files  = sorted(glob.glob(TEST_GLOB))

print("Train files:", len(train_files))
print("Validation files:", len(valid_files))
print("Test files:", len(test_files))

if not train_files or not valid_files or not test_files:
    raise FileNotFoundError("Check TRAIN_GLOB / VALID_GLOB / TEST_GLOB.")

raw_datasets = load_dataset(
    "json",
    data_files={"train": train_files, "validation": valid_files, "test": test_files}
)

print(raw_datasets)
print("Columns:", raw_datasets["train"].column_names)

print("\nExample record:")
for k, v in raw_datasets["train"][0].items():
    text = str(v)
    print(f"\n[{k}]\n{text[:1500]}")

Train files: 14
Validation files: 1
Test files: 1
DatasetDict({
    train: Dataset({
        features: ['repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url', 'partition'],
        num_rows: 412178
    })
    validation: Dataset({
        features: ['repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url', 'partition'],
        num_rows: 23107
    })
    test: Dataset({
        features: ['repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url', 'partition'],
        num_rows: 22176
    })
})
Columns: ['repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url', 'partition']

Example record:

[repo]
ageitgey/face_recognition

[path]
examples/face_recognition_knn.py

[func_name]
train

[original_string]


### Dataset configuration

For the CodeSearchNet dataset provided in `dataset/`:

**1. Docstring-to-code completion mode:**
```python
PROMPT_COLUMN = "docstring"
COMPLETION_COLUMN = "original_string"  # or "code"
```

**2. Full-text Causal LM mode (default):**
```python
PROMPT_COLUMN = None
COMPLETION_COLUMN = None
FULL_TEXT_COLUMN = "original_string"  # or "code"
```


In [4]:
# 4. Validate configuration and choose splits

columns = set(raw_datasets["train"].column_names)

if PROMPT_COLUMN is not None or COMPLETION_COLUMN is not None:
    if PROMPT_COLUMN is None or COMPLETION_COLUMN is None:
        raise ValueError("Set both PROMPT_COLUMN and COMPLETION_COLUMN.")
    if PROMPT_COLUMN not in columns:
        raise KeyError(f"Missing prompt column: {PROMPT_COLUMN}")
    if COMPLETION_COLUMN not in columns:
        raise KeyError(f"Missing completion column: {COMPLETION_COLUMN}")
    DATA_MODE = "prompt_completion"
else:
    if FULL_TEXT_COLUMN not in columns:
        raise KeyError(f"Missing full-text column: {FULL_TEXT_COLUMN}")
    DATA_MODE = "full_text"

if USE_SUBSET:
    train_dataset = raw_datasets["train"].shuffle(seed=SEED).select(
        range(min(TRAIN_SAMPLES, len(raw_datasets["train"])))
    )
    valid_dataset = raw_datasets["validation"].shuffle(seed=SEED).select(
        range(min(EVAL_SAMPLES, len(raw_datasets["validation"])))
    )
    test_dataset = raw_datasets["test"].shuffle(seed=SEED).select(
        range(min(EVAL_SAMPLES, len(raw_datasets["test"])))
    )
else:
    train_dataset = raw_datasets["train"]
    valid_dataset = raw_datasets["validation"]
    test_dataset = raw_datasets["test"]

print("DATA_MODE:", DATA_MODE)
print("Train:", len(train_dataset), "Validation:", len(valid_dataset), "Test:", len(test_dataset))

DATA_MODE: full_text
Train: 20000 Validation: 2000 Test: 2000


In [5]:
# 5. Tokenizer + 4-bit NF4 model

use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map={"": 0},
)

model.config.use_cache = False

print("Compute dtype:", compute_dtype)
print("Model device:", next(model.parameters()).device)

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Compute dtype: torch.bfloat16
Model device: cuda:0


In [6]:
# 6. Prepare QLoRA and attach LoRA to all linear layers

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, peft_config)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

model.config.use_cache = False

model.print_trainable_parameters()

trainable params: 47,677,440 || all params: 3,078,048,768 || trainable%: 1.5490


In [7]:
# 7. Tokenization
# Prompt tokens get label=-100 so loss is computed only on completion tokens.
# Full text appends eos_token_id so model learns function boundaries.

def tokenize_prompt_completion(example):
    prompt = "" if example[PROMPT_COLUMN] is None else str(example[PROMPT_COLUMN])
    completion = "" if example[COMPLETION_COLUMN] is None else str(example[COMPLETION_COLUMN])

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]
    completion_ids = completion_ids + [tokenizer.eos_token_id]

    if len(prompt_ids) + len(completion_ids) > MAX_LENGTH:
        if len(completion_ids) >= MAX_LENGTH:
            completion_ids = completion_ids[:MAX_LENGTH]
            prompt_ids = []
        else:
            available_for_prompt = MAX_LENGTH - len(completion_ids)
            prompt_ids = prompt_ids[-available_for_prompt:]

    input_ids = prompt_ids + completion_ids
    labels = [-100] * len(prompt_ids) + completion_ids.copy()

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


def tokenize_full_text(example):
    text = "" if example[FULL_TEXT_COLUMN] is None else str(example[FULL_TEXT_COLUMN])

    input_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    input_ids = input_ids[: MAX_LENGTH - 1] + [tokenizer.eos_token_id]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": input_ids.copy(),
    }


tokenize_function = tokenize_prompt_completion if DATA_MODE == "prompt_completion" else tokenize_full_text

tokenized_train = train_dataset.map(
    tokenize_function, remove_columns=train_dataset.column_names, desc="Tokenizing train"
)
tokenized_valid = valid_dataset.map(
    tokenize_function, remove_columns=valid_dataset.column_names, desc="Tokenizing validation"
)
tokenized_test = test_dataset.map(
    tokenize_function, remove_columns=test_dataset.column_names, desc="Tokenizing test"
)

sample = tokenized_train[0]
print("Token count:", len(sample["input_ids"]))
print("Supervised token count:", sum(x != -100 for x in sample["labels"]))

if DATA_MODE == "prompt_completion":
    target_ids = [t for t, y in zip(sample["input_ids"], sample["labels"]) if y != -100]
    print("\nTarget completion preview:")
    print(tokenizer.decode(target_ids, skip_special_tokens=False)[:2000])


Token count: 161
Supervised token count: 161


In [9]:
# 8. Collator + Trainer

import inspect, dataclasses, re

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

# Build TrainingArguments dynamically to support any transformers version
sig_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
if dataclasses.is_dataclass(TrainingArguments):
    sig_params.update(f.name for f in dataclasses.fields(TrainingArguments))

args_dict = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": NUM_EPOCHS,

    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,

    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "lr_scheduler_type": "cosine",

    "logging_strategy": "steps",
    "logging_steps": LOGGING_STEPS,

    "save_strategy": "steps",
    "save_steps": SAVE_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,

    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,

    "bf16": use_bf16,
    "fp16": not use_bf16,

    "optim": "paged_adamw_8bit",

    "gradient_checkpointing": True,
    "max_grad_norm": 0.3,

    "dataloader_num_workers": 2,
    "group_by_length": True,

    "report_to": "none",
    "remove_unused_columns": False,
    "save_safetensors": True,
}

# Handle eval_strategy vs evaluation_strategy
if "eval_strategy" in sig_params:
    args_dict["eval_strategy"] = "steps"
    args_dict["eval_steps"] = EVAL_STEPS
else:
    args_dict["evaluation_strategy"] = "steps"
    args_dict["eval_steps"] = EVAL_STEPS

# Handle warmup_ratio vs warmup_steps
if "warmup_ratio" in sig_params:
    args_dict["warmup_ratio"] = WARMUP_RATIO
else:
    args_dict["warmup_steps"] = 10

# Handle gradient_checkpointing_kwargs
if "gradient_checkpointing_kwargs" in sig_params:
    args_dict["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

# Filter out parameters not in signature/dataclass fields
valid_args = {k: v for k, v in args_dict.items() if k in sig_params}

# Instantiate TrainingArguments with safe retry fallback
training_args = None
while training_args is None and valid_args:
    try:
        training_args = TrainingArguments(**valid_args)
    except TypeError as e:
        match = re.search(r"unexpected keyword argument '([^']+)'", str(e))
        if match:
            bad_key = match.group(1)
            print(f"Removing unsupported argument for your transformers version: '{bad_key}'")
            valid_args.pop(bad_key, None)
        else:
            raise e

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
)

print("Effective batch size:",
      PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)


Effective batch size: 16


In [10]:
# 9. Train

torch.cuda.empty_cache()

print("Starting QLoRA fine-tuning...")
result = trainer.train()

print("\nTraining complete.")
print("Global steps:", result.global_step)
print("Training loss:", result.training_loss)

Starting QLoRA fine-tuning...


Step,Training Loss,Validation Loss
100,1.575891,1.427095
200,1.464852,1.425404
300,1.481845,1.425340
400,1.414017,1.423829
500,1.463285,1.418112
600,1.367500,1.416637
700,1.324674,1.415275
800,1.389960,1.418925
900,1.379348,1.414304
1000,1.434669,1.411582



Training complete.
Global steps: 2500
Training loss: 1.3797875045776367


In [11]:
# 10. Save LoRA adapter + tokenizer

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved adapter:", ADAPTER_DIR)

Saved adapter: /home/mastermind/Desktop/extras/CodeCompletionModel/saved_models/starcoder2-3b-qlora-code-completion


In [12]:
# 11. Held-out test loss + perplexity

test_results = trainer.evaluate(eval_dataset=tokenized_test)
test_loss = test_results.get("eval_loss")

perplexity = (
    math.exp(test_loss)
    if test_loss is not None and math.isfinite(test_loss)
    else float("nan")
)

print("===== TEST RESULTS =====")
print(f"Test loss:  {test_loss:.6f}")
print(f"Perplexity: {perplexity:.6f}")

Training Loss,Validation Loss,Step
1.286062,1.425824,2500


===== TEST RESULTS =====
Test loss:  1.425824
Perplexity: 4.161285


In [13]:
# 12. Completion-only inference

def generate_code_completion(
    prompt_text,
    max_new_tokens=150,
    temperature=0.2,
    top_p=0.95,
):
    was_training = model.training
    model.eval()
    
    # Temporarily enable KV cache for efficient inference generation
    model.config.use_cache = True

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "use_cache": True,
    }

    if temperature > 0:
        kwargs.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": top_p,
        })
    else:
        kwargs["do_sample"] = False

    with torch.inference_mode():
        outputs = model.generate(**inputs, **kwargs)

    prompt_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][prompt_length:]

    completion = tokenizer.decode(generated_ids, skip_special_tokens=True)

    if was_training:
        model.train()
        model.config.use_cache = False

    return completion


prompts = [
    """def calculate_factorial(n):
    \"\"\"Calculates the factorial of a non-negative integer n.
    \"\"\"""",
    """def is_prime(n):
    \"\"\"Return True if n is a prime number.
    \"\"\"""",
    """def reverse_string(text):
    \"\"\"Reverse a string and return the result.
    \"\"\"""",
]

for i, prompt in enumerate(prompts, 1):
    print("=" * 80)
    print("PROMPT", i)
    print("=" * 80)
    print(prompt)
    print("\n--- COMPLETION ---")
    print(generate_code_completion(prompt, max_new_tokens=100, temperature=0.2))
    print()


PROMPT 1
def calculate_factorial(n):
    """Calculates the factorial of a non-negative integer n.
    """

--- COMPLETION ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



    if n < 0:
        raise ValueError("Cannot calculate factorial of negative number.")
    elif n == 0:
        return 1
    else:
        return n * calculate_factorial(n - 1)

PROMPT 2
def is_prime(n):
    """Return True if n is a prime number.
    """

--- COMPLETION ---

    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, int(math.sqrt(n)) + 1, 2):
        if n % i == 0:
            return False
    return True

PROMPT 3
def reverse_string(text):
    """Reverse a string and return the result.
    """

--- COMPLETION ---

    return ''.join(reversed(text))



## 13. Recommended experiment sequence

**Pipeline test**

```text
USE_SUBSET = True
TRAIN_SAMPLES = 2_000
EVAL_SAMPLES = 500
NUM_EPOCHS = 1
MAX_LENGTH = 512
```

**Prototype**

```text
USE_SUBSET = True
TRAIN_SAMPLES = 20_000
EVAL_SAMPLES = 2_000
NUM_EPOCHS = 2
MAX_LENGTH = 1024
```

**Final run**

```text
USE_SUBSET = False
NUM_EPOCHS = 2
MAX_LENGTH = 1024
```

If VRAM is insufficient, reduce `MAX_LENGTH` before changing the LoRA configuration.

For code completion, the most important issue is having the correct **prompt → completion** training boundary. Perplexity alone is not enough; add syntax checks, exact match, or unit-test/pass-rate evaluation when your dataset permits it.

In [14]:
# 14. Final diagnostics

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print("Total parameters:", f"{total:,}")
print("Trainable parameters:", f"{trainable:,}")
print("Trainable percentage:", f"{100 * trainable / total:.4f}%")
print("Peak VRAM:", round(torch.cuda.max_memory_allocated() / 1024**3, 2), "GB")
print("Adapter directory:", ADAPTER_DIR)

Total parameters: 1,638,878,208
Trainable parameters: 47,677,440
Trainable percentage: 2.9092%
Peak VRAM: 3.63 GB
Adapter directory: /home/mastermind/Desktop/extras/CodeCompletionModel/saved_models/starcoder2-3b-qlora-code-completion
